# 02 — Data Structures
### dict, list, set, tuple, and comprehensions

The whole project passes data around almost entirely as **dicts** (a ticket's
classification, a customer's record, a case report) and **lists of dicts**
(order history, retrieved KB chunks). This notebook covers the structures
themselves, then the comprehension syntax the project uses to transform
them concisely.

## 2.1 Dictionaries — the project's default data shape

A dict maps keys to values. Almost every function in this project takes a
dict in and returns a dict out, rather than a custom class — that's a
deliberate simplicity choice for the non-LangChain version (the Pydantic
models exist, but plain dicts flow between the pipeline steps).

In [ ]:
customer = {
    "customer_id": "CUST001",
    "name": "Ananya Rao",
    "account_standing": "good",
}

print(customer["name"])            # direct access -- raises KeyError if missing
print(customer.get("vip_tier"))    # .get() returns None instead of raising
print(customer.get("vip_tier", "not set"))  # .get() with a default value


Ananya Rao
None
not set


Compare to the real `database.get_customer()`:

```python
def get_customer(customer_id: str) -> dict | None:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    row = conn.execute("SELECT * FROM customers WHERE customer_id = ?", (customer_id,)).fetchone()
    conn.close()
    return dict(row) if row else None
```

The last line converts a database row object into a plain dict — that's
why every caller downstream can just use `customer["name"]` or
`customer.get(...)` without knowing anything about SQLite.

## 2.2 Lists, and lists of dicts

An order history is a list of dicts — one dict per order.

In [ ]:
orders = [
    {"order_id": "ORD0001", "amount": 450.0, "status": "delivered"},
    {"order_id": "ORD0002", "amount": 6500.0, "status": "in_transit"},
]

for o in orders:
    print(f"{o['order_id']}: \u20b9{o['amount']:.2f} ({o['status']})")

# Find the highest-value order -- a common pattern in agents.py / pipeline.py
most_expensive = max(orders, key=lambda o: o["amount"])
print("\nMost expensive:", most_expensive)


ORD0001: ₹450.00 (delivered)
ORD0002: ₹6500.00 (in_transit)


`max(orders, key=lambda o: o["amount"])` is worth pausing on: `key=` tells
`max()` *what to compare*, not what to return — it still returns the full
dict, just chosen by comparing each one's `"amount"` value. This
`lambda`-as-sort/filter-key pattern shows up again in `retrieval.py`.

## 2.3 List comprehensions

A list comprehension builds a new list by transforming or filtering an
existing one, in one line instead of a `for` loop with `.append()`.

In [ ]:
amounts = [o["amount"] for o in orders]
print(amounts)

high_value = [o for o in orders if o["amount"] > 1000]
print(high_value)

labels = [f"{o['order_id']} (\u20b9{o['amount']:.0f})" for o in orders]
print(labels)


Compare to the real line in `pipeline.py`:

```python
"kb_chunks_used": [f"{c['source_doc']}#{c['section']}" for c in kb_chunks],
```

Same shape as `labels` above: take a list of dicts, produce a list of
formatted strings, one per item.

## 2.4 Sets — fast membership checks

`escalation_rules.py` defines:

```python
HARD_ESCALATE_ISSUE_TYPES = {
    "fraud_suspected",
    "complaint_escalation",
}
```

That's a **set literal** (curly braces, no `key: value` pairs — that's what
makes it a set instead of a dict). Sets are for fast "is this value in this
collection?" checks — `issue_type in HARD_ESCALATE_ISSUE_TYPES` is O(1),
while checking against a list would be O(n). For 2 items it doesn't matter
for performance, but it correctly signals to a reader "this is a collection
of unique category labels I'm checking membership against," not an ordered
sequence.

In [ ]:
HARD_ESCALATE_ISSUE_TYPES = {"fraud_suspected", "complaint_escalation"}

for issue_type in ["fraud_suspected", "order_status", "complaint_escalation"]:
    print(issue_type, "->", issue_type in HARD_ESCALATE_ISSUE_TYPES)


## 2.5 Tuples — fixed-size, ordered, often used for multiple return values

`decide_escalation`'s helper functions return `tuple[bool, str]` — a
2-element tuple that Python lets you unpack directly into two variables.

In [ ]:
def check_confidence(confidence: float, threshold: float = 0.75) -> tuple[bool, str]:
    if confidence < threshold:
        return True, f"confidence {confidence:.2f} is below threshold {threshold}"
    return False, "passed"

should_escalate, reason = check_confidence(0.4)
print(should_escalate)
print(reason)


## 2.6 Dict comprehensions and unpacking dicts

Dict comprehensions build a dict the same way list comprehensions build a
list. `**dict` unpacking spreads a dict's key-value pairs as keyword
arguments into a function call or into another dict literal — this is
exactly how `pipeline.py` builds `TicketClassification(**classification)`.

In [ ]:
order_lookup = {o["order_id"]: o["amount"] for o in orders}
print(order_lookup)

def describe_order(order_id: str, amount: float, status: str = "unknown") -> str:
    return f"{order_id}: \u20b9{amount:.2f} [{status}]"

order_dict = {"order_id": "ORD0002", "amount": 6500.0, "status": "in_transit"}
print(describe_order(**order_dict))   # spreads the dict as keyword arguments


## Exercise

1. Given this list of tickets:
```python
tickets = [
    {"ticket_id": "T001", "issue_type": "refund_request", "confidence": 0.9},
    {"ticket_id": "T002", "issue_type": "fraud_suspected", "confidence": 0.6},
    {"ticket_id": "T003", "issue_type": "order_status", "confidence": 0.95},
]
```
   Write a list comprehension that returns just the `ticket_id`s where
   `confidence` is below `0.75`.
2. Write a set of "always-escalate" issue types of your own choosing, and a
   one-line expression using `in` to check whether `"fraud_suspected"` is a
   member.
3. Open `pipeline.py` in the project folder and find one more dict or list
   comprehension. Rewrite it as an equivalent `for` loop with `.append()` to
   confirm you understand exactly what it's doing.